In [0]:
import os
import sys
import importlib
import utils.custom_utils
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    when,
    round,
    max,
    current_timestamp
)
from delta.tables import DeltaTable
from utils.gold import GoldAnalytics
from utils.logger import logger

In [0]:
#Reading Data from Delta Tables
df_customers = spark.read.table("ride_analytics.silver.customers")
df_drivers = spark.read.table("ride_analytics.silver.drivers")
df_locations = spark.read.table("ride_analytics.silver.locations")
df_payments = spark.read.table("ride_analytics.silver.payments")
df_trips = spark.read.table("ride_analytics.silver.trips")
df_vehicles = spark.read.table("ride_analytics.silver.vehicles")


In [0]:
#Creating a Object
gold_obj = GoldAnalytics()

### GOLD TABLE 1 — Daily Trip KPI

In [0]:
# Check whether Gold table already exists
gold_table = "ride_analytics.gold.daily_trip_kpis"

logger.info("Checking Gold table: %s", gold_table)

if not spark.catalog.tableExists(gold_table):

    # First run: perform full aggregation
    logger.info("Starting FULL LOAD for Daily Trip KPIs")

    daily_trip_kpis = (
        gold_obj.calculate_daily_trip_kpis(df_trips)
    )

    # Create the Gold table
    daily_trip_kpis.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(gold_table)

    logger.info(
        "Gold table created successfully: %s",
        gold_table
    )

else:

    # Subsequent runs: process only new/updated data
    logger.info("Starting INCREMENTAL LOAD for Daily Trip KPIs")

    gold_obj.incremental_daily_trip_kpis(
        df_trips,
        gold_table
    )

    logger.info(
        "Incremental Daily Trip KPI processing completed"
    )

2026-09-10 19:32:02,242 - INFO - Checking Gold table: ride_analytics.gold.daily_trip_kpis
2026-09-10 19:32:02,407 - INFO - Starting FULL LOAD for Daily Trip KPIs
2026-09-10 19:32:02,409 - INFO - Daily KPIs calculated
2026-09-10 19:32:05,587 - INFO - Gold table created successfully: ride_analytics.gold.daily_trip_kpis


### GOLD TABLE 2 — Driver Performance

In [0]:
# Join trips with driver information
driver_trip_df = (
    df_trips.alias("t")
    .join(
        df_drivers.alias("d"),
        col("t.driver_id") == col("d.driver_id"),
        "left"
    )
)

In [0]:
gold_table = "ride_analytics.gold.driver_performance"

# Check whether Gold table already exists
logger.info("Checking Gold table: %s", gold_table)

if not spark.catalog.tableExists(gold_table):

    # First run → calculate all drivers
    logger.info("Starting FULL LOAD for Driver Performance")

    driver_performance = (
        gold_obj.calculate_driver_performance(
            driver_trip_df
        )
    )

    logger.info("Driver Performance KPI calculation completed")

    # Create Gold table
    driver_performance.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(gold_table)

    logger.info(
        "Gold table created successfully: %s",
        gold_table
    )

else:

    # Subsequent runs → incremental processing
    logger.info("Starting INCREMENTAL LOAD for Driver Performance")

    gold_obj.incremental_driver_performance(
        df_trips,
        df_drivers,
        gold_table
    )

    logger.info(
        "Incremental Driver Performance processing completed"
    )

2026-09-10 19:32:06,447 - INFO - Checking Gold table: ride_analytics.gold.driver_performance
2026-09-10 19:32:06,613 - INFO - Starting FULL LOAD for Driver Performance
2026-09-10 19:32:06,616 - INFO - Driver Performance KPI calculation completed
2026-09-10 19:32:10,203 - INFO - Gold table created successfully: ride_analytics.gold.driver_performance


### GOLD TABLE 3 — Customer Analysis

In [0]:
# Join customers with their trip information
customer_trip_df = (
    df_customers.alias("c")
    .join(
        df_trips.alias("t"),
        col("c.customer_id") == col("t.customer_id"),
        "left"
    )
)

In [0]:
gold_table = "ride_analytics.gold.customer_analysis"

# Check whether Customer Analysis Gold table exists
logger.info("Checking Gold table: %s", gold_table)

if not spark.catalog.tableExists(gold_table):

    # First run → calculate all customers
    logger.info("Starting FULL LOAD for Customer Analysis")

    customer_analysis = (
        gold_obj.calculate_customer_analysis(
            customer_trip_df
        )
    )

    logger.info("Customer KPI calculation completed")

    # Create Gold table
    customer_analysis.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(gold_table)

    logger.info(
        "Gold table created successfully: %s",
        gold_table
    )

else:

    # Subsequent runs → incremental processing
    logger.info("Starting INCREMENTAL LOAD for Customer Analysis")

    gold_obj.incremental_customer_analysis(
        df_trips,
        df_customers,
        gold_table
    )

    logger.info(
        "Incremental Customer Analysis processing completed"
    )

2026-09-10 19:32:11,041 - INFO - Checking Gold table: ride_analytics.gold.customer_analysis
2026-09-10 19:32:11,198 - INFO - Starting FULL LOAD for Customer Analysis
2026-09-10 19:32:11,200 - INFO - Customer KPI calculation completed
2026-09-10 19:32:14,890 - INFO - Gold table created successfully: ride_analytics.gold.customer_analysis


### GOLD TABLE 4 — Payment Analysis

In [0]:
gold_table = "ride_analytics.gold.payment_analysis"

# Check whether Gold table already exists
logger.info("Checking Gold table: %s", gold_table)

if not spark.catalog.tableExists(gold_table):

    # First run → calculate all payment groups
    logger.info("Starting FULL LOAD for Payment Analysis")

    payment_analysis = (
        gold_obj.calculate_payment_analysis(
            df_payments
        )
    )

    logger.info("Payment KPI calculation completed")

    # Create Gold table
    payment_analysis.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(gold_table)

    logger.info(
        "Gold table created successfully: %s",
        gold_table
    )

else:
    # First run → calculate all payment groups
    logger.info("Started - Payment Analysis Update")

    new_payment_analysis = (
        gold_obj.calculate_payment_analysis(
            df_payments
        )
    )

    logger.info("Payment KPI calculation completed")

    # Create Gold table
    new_payment_analysis.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(gold_table)

    logger.info(
        "Completed - Payment Analysis Update: %s",
        gold_table
    )

2026-09-10 19:32:15,254 - INFO - Checking Gold table: ride_analytics.gold.payment_analysis
2026-09-10 19:32:15,412 - INFO - Starting FULL LOAD for Payment Analysis
2026-09-10 19:32:15,414 - INFO - Payment KPI calculation completed
2026-09-10 19:32:18,825 - INFO - Gold table created successfully: ride_analytics.gold.payment_analysis


- END